# sar-flood-shift: end-to-end run
Does a SAR flood-mapping model know when it is out of its depth?

**Runtime → Change runtime type → T4 GPU.** Total time: roughly 30–45 minutes, most of it training the 5-member ensemble.

In [ ]:
# Optional: keep checkpoints and results in Google Drive so a disconnect doesn't cost you the training run
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/sar-flood-shift-work'
else:
    WORK = '/content/work'
import os; os.makedirs(WORK, exist_ok=True); print(WORK)

In [ ]:
!git clone -q https://github.com/Zubair-DLCS/sar-flood-shift.git /content/sar-flood-shift || (cd /content/sar-flood-shift && git pull -q)
%cd /content/sar-flood-shift
!pip install -q -r requirements.txt

In [ ]:
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Data: Sen1Floods11 hand-labelled (~1 GB)

In [ ]:
!python scripts/download_sen1floods11.py --out data/sen1floods11
!ls data/sen1floods11/S1Hand | wc -l; wc -l data/sen1floods11/splits/*.csv

## 2. Train a 5-member deep ensemble
Skips members whose checkpoint already exists in Drive, so you can resume after a disconnect.

In [ ]:
import os, subprocess
RUNS = f'{WORK}/runs/ensemble'
for seed in range(5):
    if os.path.exists(f'{RUNS}/member_{seed}.pt') and os.path.exists(f'{RUNS}/done_{seed}'):
        print('member', seed, 'already trained'); continue
    !python scripts/train.py --data data/sen1floods11 --out {RUNS} --members 1 --first-seed {seed} --epochs 60
    open(f'{RUNS}/done_{seed}', 'w').close()

## 3. Evaluate on seen countries vs Bolivia

In [ ]:
RES = f'{WORK}/results'
!python scripts/evaluate.py --data data/sen1floods11 --runs {RUNS} --out {RES}
from IPython.display import Markdown, Image, display
display(Markdown(open(f'{RES}/table.md').read()))
for f in ['reliability', 'risk_coverage', 'uncertainty_vs_error', 'per_country', 'worst_chips']:
    display(Image(f'{RES}/figures/{f}.png'))

## 4. Case study: Sindh, Pakistan, 2022
Pulls a pre-event and a flood Sentinel-1 RTC scene (same orbit) from Microsoft Planetary Computer. No account is needed.
To change the area, pass `--bbox lon_min lat_min lon_max lat_max`.

In [ ]:
PK = f'{WORK}/data/pakistan'
!python scripts/pakistan_case_study.py fetch --out {PK}

Optional ground truth: download a UNOSAT flood-extent shapefile for the same area and dates, upload it, and set `REF` to its path.
Without it, the script compares against an Otsu threshold, which is only a sanity check, not ground truth.

In [ ]:
REF = ''   # e.g. '/content/drive/MyDrive/unosat/FL20220830PAK.shp'
ref_arg = f'--reference {REF}' if REF else ''
!python scripts/pakistan_case_study.py predict --runs {RUNS} --data {PK} --out {RES}/pakistan --sen1floods11-metrics {RES}/metrics.json {ref_arg}
display(Image(f'{RES}/pakistan/pakistan_maps.png')); display(Image(f'{RES}/pakistan/shift_summary.png'))
print(open(f'{RES}/pakistan/metrics.json').read())

## 5. Copy results into the repo for the README
Commit `results/` (figures + JSON; the GeoTIFFs are git-ignored), then fill in the Results and "Where it breaks" sections of the README.

In [ ]:
!mkdir -p results && cp -r {RES}/* results/ && ls -R results | head -40